In [ ]:
import mpmath as mp
from scipy import optimize
import math
import numpy as np
from scipy.io import savemat

In [ ]:
def f_Z(l, x):
    """ appendix F remark 12
    Compute the function f_Z(l, x) defined from the original equation:
        f_l(x) = x*J_{l-1/2}(x) - (l+1)*J_{l+1/2}(x),
    rewritten as:
        f_Z(l, x) = x*J_{l-1/2}(x)/J_{l+1/2}(x) - (l+1).
    
    The roots of this equation are important for eigenvalue problems
    involving spherical Bessel functions with modified boundary conditions.
    """
    return x*mp.besselj(l - 0.5, x) / mp.besselj(l + 0.5, x) - (l + 1)

def f_Y(l, x):
    """appendix F remark 12
    Compute the function f_Y(l, x), derived from the original equation:
        f_l(x) = x*J_{l-1/2}(x) - (l+2)*J_{l+1/2}(x)
                 + (x^2 - (l-1)l)(x^2 - (l-1)(l+2)) / (x^2 - (l-1)l(l+2)) * J_{l+1/2}(x)

    In practice this is rewritten as:
        For l > 1:
            f_Y(l, x) = x*J_{l-1/2}(x)/J_{l+1/2}(x) - (l+2)
                        + ((x^2 - (l-1)l) * (x^2 - (l-1)(l+2)))
                          / (x^2 - (l-1)l(l+2))
        For l = 1:
            f_Y(1, x) = x*J_{1/2}(x)/J_{3/2}(x) - 3 + x^2

    The roots of this equation are important for eigenvalue problems
    involving spherical Bessel functions with modified boundary conditions.
    """
    if l > 1:
        return x*mp.besselj(l - 0.5, x) / mp.besselj(l + 0.5, x) - (l + 2) +\
               (x**2 - (l - 1) * l) * (x**2 - (l - 1) * (l + 2)) / (x**2 - (l - 1) * l * (l + 2))
    else:
        return x*mp.besselj(l - 0.5, x) / mp.besselj(l + 0.5, x) - (l + 2) + x**2

def build_intervals(l, n, gamma=None):
    """
    Construct n bracketing intervals for root finding based on consecutive zeros of J_{l+1/2}(x).

    Interval scheme:
        - First interval: (δ, j_{l+1/2,1}-δ)
        - Subsequent intervals: (j_{l+1/2,m-1}+δ, j_{l+1/2,m}-δ) for m = 2, ..., n
      where j_{ν,m} denotes the m-th positive zero of J_ν(x) and δ is a small positive padding
      used to avoid evaluating at singular points or exactly at zeros.

    If a singular/critical point `gamma` lies inside (a, b), the interval is split into
    two sub-intervals (a+δ, gamma-δ) and (gamma+δ, b-δ) to keep the function well-behaved.

    Args:
        l (int): index of Bessel functions in J_{l+1/2}.
        n (int): Number of intervals to construct.
        gamma (float or None): Optional interior point to exclude, representing a singularity.

    Returns:
        list[tuple[float, float]]: A list of open intervals (a, b) suitable for bracketing.

    Notes:
        - Uses mpmath.besseljzero(l + 1/2, m) to retrieve j_{l+1/2,m}.
        - The small padding `delta = 1e-6` prevents evaluating at endpoints.
    """
    b = 0
    intervals = []
    for m in range(n):
        a = b
        b = float(mp.besseljzero(l + 0.5, m + 1))
        delta = 1e-6
        if gamma is not None and a < gamma < b:
            intervals.append((a+delta, gamma-delta))
            intervals.append((gamma+delta, b-delta))
        else:
            intervals.append((a+delta, b-delta))
    return intervals


def solve_roots_mixed(l, n, fun, vari, rtol=1e-12):
    """
    Find the first n roots of f_l(x) = 0 by bracketing on intervals determined from J_{l+1/2} zeros
    and solving within each bracket via Brent's method.

    First-interval rule (regularization):
        - If vari == "Y" and l > 1, define a critical point gamma = sqrt(l(l-1)(l+2)) and split brackets
          around it (handled in build_intervals). For vari == "Y" and l == 1, the first root is set to 0 by convention.
        - Otherwise, standard bracketing with f_l on all intervals.

    For each interval (left, right), this function:
        1) Checks a sign change f(left)*f(right) <= 0 to ensure a valid bracket.
        2) Calls `scipy.optimize.root_scalar(..., method="brentq")` with the given tolerance.

    Args:
        l (int): Angular momentum–like index.
        n (int): Number of roots to locate.
        fun (callable): Function of x (with l captured) whose roots are sought.
        vari (str): Variant flag (e.g., "Y" or "Z") to control special handling.
        rtol (float): Relative tolerance passed to Brent's method.

    Returns:
        tuple:
            - roots (list[float]): The roots found, in ascending order across brackets.
            - intervals (list[tuple[float, float]]): The bracketing intervals actually used.

    Notes:
        - Asserts f(left)*f(right) <= 0; if violated, tighten brackets upstream.
        - For (vari == "Y" and l == 1) on the very first interval, a root at 0 is appended by convention.
    """
    # For vari "Y" and l > 1, compute the singularity point gamma
    gamma = np.sqrt(l*(l-1)*(l+2)) if vari == "Y" and l > 1 else None
    intervals = build_intervals(l, n, gamma)
    roots = []
    
    for i, (left, right) in enumerate(intervals):
        # Special case: for vari "Y" and l=1, add root at 0 by convention
        if vari == "Y" and l == 1 and i == 0:
            roots.append(0)

        # Ensure a sign change in the bracket
        fL, fR = fun(left), fun(right)
        assert fL*fR <= 0

        # If we have a valid bracket, proceed with root finding
        res = optimize.root_scalar(lambda x: float(fun(x)),
                                   bracket=(left, right),
                                   method='brentq', rtol=rtol)
        roots.append(res.root)

    return roots, intervals

def index_approx(nu, x):
    """
    Approximate the index (count) of J_ν zeros not exceeding x using the standard asymptotic phase formula.
    Used as an estimate for the number of roots to search for.

    Uses:
        θ(x) = sqrt(x^2 - ν^2) - ν * arccos(ν/x),  for x ≥ ν,
    and estimates the index as:
        idx ≈ floor( θ(x)/π + 1/4 ) + 1.

    If ν > x, return 1 as a conservative lower bound on the first zero index to consider.

    Args:
        nu (float): Order ν of the Bessel function J_ν.
        x (float): Cutoff point on the positive real axis.

    Returns:
        int: Approximate zero index at/below x (≥ 1).
    """
    if nu > x:
        return 1
    th = np.sqrt(x*x - nu*nu) - nu*np.arccos(nu/x)
    return math.ceil(np.floor(th/np.pi + 0.25)) + 1

def roots_for_l_below_R(l, R, fun, vari, rtol=1e-12):
    """
    Compute all roots of f_l(x) below a cutoff R (>0).

    Procedure:
        1) Use `index_approx(l+1/2, R)` to estimate how many J_{l+1/2} zeros lie below R.
        2) Increase N until j_{l+1/2,N} ≥ R to ensure coverage of all brackets under R.
        3) Call `solve_roots_mixed` on N intervals and drop any roots ≥ R.

    Args:
        l (int): Angular momentum–like index.
        R (float): Positive cutoff; return only roots x with x < R.
        fun (callable): Function of x (with l captured) whose roots are sought.
        vari (str): Variant flag (passed to the mixed solver).
        rtol (float): Relative tolerance for Brent's method.

    Returns:
        list[float]: Sorted list of roots strictly below R.
    """
    N = max(index_approx(l + 0.5, R), 1)
    while mp.besseljzero(l + 0.5, N) < R:
        N += 1
    root_lst, _ = solve_roots_mixed(l, N, fun, vari, rtol)
    while root_lst and root_lst[-1] >= R:
        root_lst.pop()
    return root_lst

def find_all_roots_below_R(R, fun, vari, rtol=1e-12, mp_dps=50):
    """
    Find all roots k < R of f_l(k) = 0 for integer l starting from 1 upward.

    For each l = 1, 2, ..., ceil(R):
        - Compute all roots below R via `roots_for_l_below_R`.
        - If no roots are found for some l, stop early (since j_{l+1/2,1} grows roughly linearly in l).

    Args:
        R (float): Positive cutoff for roots.
        fun (callable): Function f(l, x) taking (l, x) and returning a scalar.
        vari (str): Variant flag to pass through the pipeline (e.g., "Y"/"Z").
        rtol (float): Relative tolerance for the root solver.
        mp_dps (int): mpmath decimal precision to use.

    Returns:
        list[list[float]]: A list whose i-th element (0-based) holds the roots for l = i+1.
                           Each inner list is sorted ascending and contains only roots < R.

    Notes:
        - The previously stated return type as a dict is not accurate for this implementation;
          this function returns a list of lists.
    """
    mp.mp.dps = mp_dps
    out = []
    for l in range(1, math.ceil(R) + 1):
        ks = roots_for_l_below_R(l, R, lambda x: fun(l, x), vari, rtol)
        if ks:
            out.append(ks)
        else:
            break
    return out


In [ ]:
# Calculate the roots for function f_Z
result_Z = find_all_roots_below_R(100, f_Z, "Z", rtol=1e-15, mp_dps=50)

In [ ]:
# Calculate the roots for function f_Y
result_Y = find_all_roots_below_R(100, f_Y, "Y", rtol=1e-15, mp_dps=50)

In [ ]:
# Convert the results to padded numpy arrays for saving
result_Z_padded = np.zeros((len(result_Z), len(result_Z[0])), dtype=np.float64)
for i, flist in enumerate(result_Z):
    result_Z_padded[i, :len(flist)] = np.array(flist, dtype=np.float64)
result_Y_padded = np.zeros((len(result_Y), len(result_Y[0])), dtype=np.float64)
for i, flist in enumerate(result_Y):
    result_Y_padded[i, :len(flist)] = np.array(flist, dtype=np.float64)

# Save to a .mat file
savemat("../data/bessel_zeros.mat", {"Y_zeros": result_Y_padded, "Z_zeros": result_Z_padded})